# FT-Transformer Search

This notebook runs a higher-capacity FT-transformer search on the frozen development folds.

It keeps the same trimmed sequence scope used in the earlier neural benchmark, but searches stronger transformer configurations with:

- larger token width
- deeper encoder stacks
- BCE vs focal loss
- scheduler-driven training
- threshold tuning on pooled out-of-fold predictions

The benchmark to beat remains the earlier best PR AUC from the repo.

## Research-Question Framing

To answer `docs/WEC2026_Problem_description.md` cleanly, this notebook uses two aligned views of model output:

- **Public contest output:** probability that the player scores later in the match
- **Internal Poisson interpretation:** expected post-checkpoint scoring intensity `lambda`

For Poisson configurations, the model predicts `log(lambda)`. That intensity is then converted to scoring probability with:

- `p(score later) = 1 - exp(-lambda)`

This keeps the final outputs aligned with the research questions while still allowing intensity-based interpretation for feature effects and event-rate reasoning.

In [1]:
from __future__ import annotations

import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / "data").exists())
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts" / "ft_transformer_search"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "scored_after"
DEVICE = torch.device("cpu")

SEQ15M_TRIMMED_FEATURES = [
    "seq15m_pressure_total",
    "seq15m_pressure_turnover_total",
    "seq15m_longest_active_streak",
    "seq15m_recency_weighted_pressure_total",
    "seq15m_active_minute_share",
    "seq15m_late_burst_count",
    "seq15m_shots_same_or_next_min_after_pressure",
]
SEQPOS_TRIMMED_FEATURES = [
    "seqpos_strict_link_rate",
    "seqpos_mean_shots_per_possession",
    "seqpos_possessions_with_shots_share",
    "seqpos_pressure_shot_escalation_share",
    "seqpos_forward_pressure_shot_share",
    "seqpos_linked_pressure_total",
    "seqpos_mean_runs_per_possession",
]
TRIMMED_SEQUENCE_FEATURES = SEQ15M_TRIMMED_FEATURES + SEQPOS_TRIMMED_FEATURES
CATEGORICAL_COLS = ["checkpoint", "position", "formation", "minute_in_window"]

FT_CONFIGS = [
    {
        "name": "ft_repro_bce",
        "d_token": 32,
        "n_layers": 2,
        "n_heads": 4,
        "ff_mult": 4,
        "dropout": 0.10,
        "lr": 0.0010,
        "weight_decay": 1e-4,
        "loss_name": "bce",
        "gamma": 0.0,
    },
    {
        "name": "ft_wide_bce",
        "d_token": 64,
        "n_layers": 3,
        "n_heads": 8,
        "ff_mult": 4,
        "dropout": 0.10,
        "lr": 0.0008,
        "weight_decay": 1e-4,
        "loss_name": "bce",
        "gamma": 0.0,
    },
    {
        "name": "ft_wide_focal",
        "d_token": 64,
        "n_layers": 3,
        "n_heads": 8,
        "ff_mult": 4,
        "dropout": 0.12,
        "lr": 0.0008,
        "weight_decay": 2e-4,
        "loss_name": "focal",
        "gamma": 2.0,
    },
    {
        "name": "ft_deep_focal",
        "d_token": 48,
        "n_layers": 4,
        "n_heads": 6,
        "ff_mult": 4,
        "dropout": 0.15,
        "lr": 0.0006,
        "weight_decay": 2e-4,
        "loss_name": "focal",
        "gamma": 2.0,
    },
    {
        "name": "ft_wide_poisson",
        "d_token": 64,
        "n_layers": 3,
        "n_heads": 8,
        "ff_mult": 4,
        "dropout": 0.10,
        "lr": 0.0008,
        "weight_decay": 1e-4,
        "loss_name": "poisson",
        "gamma": 0.0,
    },
    {
        "name": "ft_deep_poisson",
        "d_token": 48,
        "n_layers": 4,
        "n_heads": 6,
        "ff_mult": 4,
        "dropout": 0.15,
        "lr": 0.0006,
        "weight_decay": 2e-4,
        "loss_name": "poisson",
        "gamma": 0.0,
    },
]
MAX_EPOCHS = 90
EARLY_STOPPING_PATIENCE = 16


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def metric_row(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if np.unique(y_true).size > 1 else np.nan,
        "brier_score": float(brier_score_loss(y_true, y_score)),
        "target_rate": float(y_true.mean()),
    }


def tune_threshold(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float]:
    thresholds = np.linspace(0.05, 0.50, 46)
    rows = []
    for threshold in thresholds:
        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": float(
                    balanced_accuracy_score(y_true, (np.asarray(y_score) >= threshold).astype(int))
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(["balanced_accuracy", "threshold"], ascending=[False, True]).iloc[0].to_dict()


def split_inner_indices(y: np.ndarray, frac: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    train_parts = []
    valid_parts = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        rng.shuffle(indices)
        n_valid = max(1, int(round(len(indices) * frac)))
        if len(indices) <= 2:
            n_valid = 1
        valid_parts.append(indices[:n_valid])
        train_parts.append(indices[n_valid:])
    return np.sort(np.concatenate(train_parts)), np.sort(np.concatenate(valid_parts))


def prepare_fold_data(train_df: pd.DataFrame, valid_df: pd.DataFrame) -> tuple[dict[str, torch.Tensor], dict[str, torch.Tensor], dict[str, object]]:
    excluded = {TARGET, "fixture_id", "fold", "partition_role", "scored_after_eval_key", "player_id", "jersey_number"}
    numeric_cols = [
        column
        for column in train_df.columns
        if column not in excluded
        and column not in CATEGORICAL_COLS
        and (pd.api.types.is_numeric_dtype(train_df[column].dtype) or pd.api.types.is_bool_dtype(train_df[column].dtype))
    ]
    numeric_cols = [column for column in numeric_cols if column not in []]

    medians = train_df[numeric_cols].median()
    means = train_df[numeric_cols].fillna(medians).astype(float).mean()
    stds = train_df[numeric_cols].fillna(medians).astype(float).std(ddof=0).replace(0, 1.0)

    category_maps = {}
    cardinalities = []
    for column in CATEGORICAL_COLS:
        values = train_df[column].fillna("__missing__").astype(str)
        uniques = sorted(values.unique())
        category_maps[column] = {value: idx + 1 for idx, value in enumerate(uniques)}
        cardinalities.append(len(uniques) + 1)

    def encode(frame: pd.DataFrame) -> dict[str, torch.Tensor]:
        numeric = (
            frame[numeric_cols]
            .fillna(medians)
            .astype(float)
            .sub(means, axis=1)
            .div(stds, axis=1)
            .to_numpy(dtype=np.float32)
        )
        categorical = np.column_stack(
            [
                frame[column]
                .fillna("__missing__")
                .astype(str)
                .map(category_maps[column])
                .fillna(0)
                .astype(int)
                .to_numpy()
                for column in CATEGORICAL_COLS
            ]
        )
        y = frame[TARGET].astype(int).to_numpy(dtype=np.float32)
        return {
            "numeric": torch.tensor(numeric, dtype=torch.float32),
            "categorical": torch.tensor(categorical, dtype=torch.long),
            "y": torch.tensor(y, dtype=torch.float32),
        }

    metadata = {
        "numeric_cols": numeric_cols,
        "categorical_cols": CATEGORICAL_COLS,
        "cardinalities": cardinalities,
    }
    return encode(train_df), encode(valid_df), metadata


class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight: float, gamma: float = 2.0) -> None:
        super().__init__()
        self.gamma = gamma
        self.pos_weight = torch.tensor([pos_weight], dtype=torch.float32)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        pos_weight = self.pos_weight.to(logits.device)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=pos_weight)
        probs = torch.sigmoid(logits)
        pt = torch.where(targets > 0.5, probs, 1.0 - probs)
        focal = torch.pow(1.0 - pt, self.gamma)
        return (focal * bce).mean()


class PoissonBinaryIntensityLoss(nn.Module):
    def forward(self, log_rate: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        stable_log_rate = torch.clamp(log_rate, min=-10.0, max=8.0)
        return nn.functional.poisson_nll_loss(stable_log_rate, targets, log_input=True, full=False, reduction="mean")


class NumericFeatureTokenizer(nn.Module):
    def __init__(self, n_num_features: int, d_token: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_num_features, d_token) * 0.05)
        self.bias = nn.Parameter(torch.zeros(n_num_features, d_token))
        self.column_embed = nn.Parameter(torch.randn(n_num_features, d_token) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0) + self.column_embed.unsqueeze(0)


class CategoricalFeatureTokenizer(nn.Module):
    def __init__(self, cardinalities: list[int], d_token: int) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(cardinality, d_token) for cardinality in cardinalities])
        self.column_embed = nn.Parameter(torch.randn(len(cardinalities), d_token) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = []
        for idx, embedding in enumerate(self.embeddings):
            tokens.append(embedding(x[:, idx]) + self.column_embed[idx])
        return torch.stack(tokens, dim=1)


class FTTransformerSearchModel(nn.Module):
    def __init__(self, n_num_features: int, cardinalities: list[int], d_token: int, n_layers: int, n_heads: int, ff_mult: int, dropout: float) -> None:
        super().__init__()
        self.num_tokenizer = NumericFeatureTokenizer(n_num_features, d_token)
        self.cat_tokenizer = CategoricalFeatureTokenizer(cardinalities, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ff_mult,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, d_token // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token // 2, 1),
        )

    def forward(self, numeric: torch.Tensor, categorical: torch.Tensor) -> torch.Tensor:
        num_tokens = self.num_tokenizer(numeric)
        cat_tokens = self.cat_tokenizer(categorical)
        cls = self.cls_token.expand(numeric.size(0), -1, -1)
        tokens = torch.cat([cls, num_tokens, cat_tokens], dim=1)
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0, :]).squeeze(1)


def make_loss_fn(loss_name: str, y_train: torch.Tensor, gamma: float) -> nn.Module:
    positives = max(float(y_train.sum().item()), 1.0)
    negatives = max(float(len(y_train) - y_train.sum().item()), 1.0)
    pos_weight = negatives / positives
    if loss_name == "poisson":
        return PoissonBinaryIntensityLoss()
    if loss_name == "focal":
        return FocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=gamma)
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE))


def outputs_to_probabilities(raw_output: torch.Tensor, loss_name: str) -> torch.Tensor:
    if loss_name == "poisson":
        stable_log_rate = torch.clamp(raw_output, min=-10.0, max=8.0)
        rate = torch.exp(stable_log_rate)
        return 1.0 - torch.exp(-rate)
    return torch.sigmoid(raw_output)


def predict_outputs(model: nn.Module, tensors: dict[str, torch.Tensor], loss_name: str) -> dict[str, np.ndarray]:
    model.eval()
    with torch.no_grad():
        raw_output = model(tensors["numeric"].to(DEVICE), tensors["categorical"].to(DEVICE))
        if loss_name == "poisson":
            stable_log_rate = torch.clamp(raw_output, min=-10.0, max=8.0)
            lambda_rate = torch.exp(stable_log_rate)
            goal_probability = 1.0 - torch.exp(-lambda_rate)
            return {
                "goal_probability": goal_probability.cpu().numpy(),
                "log_lambda_rate": stable_log_rate.cpu().numpy(),
                "lambda_rate": lambda_rate.cpu().numpy(),
            }
        goal_probability = torch.sigmoid(raw_output)
        nan_array = np.full(goal_probability.shape[0], np.nan, dtype=float)
        return {
            "goal_probability": goal_probability.cpu().numpy(),
            "log_lambda_rate": nan_array,
            "lambda_rate": nan_array,
        }


def train_model_for_fold(config: dict[str, object], train_tensors: dict[str, torch.Tensor], seed: int) -> nn.Module:
    set_seed(seed)
    train_idx, inner_valid_idx = split_inner_indices(train_tensors["y"].cpu().numpy(), frac=0.15, seed=seed)
    subtrain = {key: value[train_idx] for key, value in train_tensors.items()}
    inner_valid = {key: value[inner_valid_idx] for key, value in train_tensors.items()}

    model = FTTransformerSearchModel(
        n_num_features=subtrain["numeric"].shape[1],
        cardinalities=metadata_global["cardinalities"],
        d_token=int(config["d_token"]),
        n_layers=int(config["n_layers"]),
        n_heads=int(config["n_heads"]),
        ff_mult=int(config["ff_mult"]),
        dropout=float(config["dropout"]),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config["lr"]), weight_decay=float(config["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    loss_fn = make_loss_fn(str(config["loss_name"]), subtrain["y"], float(config["gamma"]))

    dataset = TensorDataset(subtrain["numeric"], subtrain["categorical"], subtrain["y"])
    loader = DataLoader(dataset, batch_size=min(256, len(dataset)), shuffle=True)

    best_state = copy.deepcopy(model.state_dict())
    best_pr_auc = -np.inf
    patience_left = EARLY_STOPPING_PATIENCE

    for _ in range(MAX_EPOCHS):
        model.train()
        for numeric_batch, categorical_batch, y_batch in loader:
            numeric_batch = numeric_batch.to(DEVICE)
            categorical_batch = categorical_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            raw_output = model(numeric_batch, categorical_batch)
            loss = loss_fn(raw_output, y_batch)
            loss.backward()
            optimizer.step()

        inner_outputs = predict_outputs(model, inner_valid, str(config["loss_name"]))
        inner_pr_auc = average_precision_score(inner_valid["y"].cpu().numpy(), inner_outputs["goal_probability"])
        scheduler.step(inner_pr_auc)
        if inner_pr_auc > best_pr_auc + 1e-4:
            best_pr_auc = float(inner_pr_auc)
            best_state = copy.deepcopy(model.state_dict())
            patience_left = EARLY_STOPPING_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    return model


In [2]:
sequence_df = pd.read_csv(DATA_DIR / "sequencing_dataset.csv")
row_folds = pd.read_csv(DATA_DIR / "splits" / "modeling_row_folds.csv", usecols=["fixture_id", "player_id", "checkpoint"])
row_folds["scored_after_eval_key"] = row_folds["fixture_id"].astype(str) + "__" + row_folds["checkpoint"].astype(str) + "__" + row_folds["player_id"].astype(str)
fixture_assignments = pd.read_csv(DATA_DIR / "splits" / "fixture_assignments.csv")

sequence_df = sequence_df.merge(row_folds[["scored_after_eval_key", "fixture_id"]], on="scored_after_eval_key", how="left", validate="one_to_one")
sequence_df = sequence_df.merge(fixture_assignments, on="fixture_id", how="left", validate="many_to_one")

if sequence_df[["fixture_id", "fold", "partition_role"]].isna().any().any():
    raise ValueError("Split metadata join failed")

development_df = sequence_df[sequence_df["partition_role"] == "development"].copy().reset_index(drop=True)
fold_ids = sorted(development_df["fold"].dropna().unique())
_, _, metadata_global = prepare_fold_data(development_df.iloc[:100].copy(), development_df.iloc[:20].copy())

print("Development rows:", len(development_df))
print("Target rate:", round(float(development_df[TARGET].mean()), 4))
print("FT configs:", [config["name"] for config in FT_CONFIGS])


Development rows: 2830
Target rate: 0.0583
FT configs: ['ft_repro_bce', 'ft_wide_bce', 'ft_wide_focal', 'ft_deep_focal', 'ft_wide_poisson', 'ft_deep_poisson']


In [3]:
fold_rows = []
oof_parts = []

for config_index, config in enumerate(FT_CONFIGS):
    print(f"Running {config['name']}")
    for fold_index, fold_id in enumerate(fold_ids):
        train_df = development_df[development_df["fold"] != fold_id].copy()
        valid_df = development_df[development_df["fold"] == fold_id].copy()
        train_tensors, valid_tensors, metadata_fold = prepare_fold_data(train_df, valid_df)
        metadata_global = metadata_fold
        model = train_model_for_fold(config, train_tensors, seed=RANDOM_STATE + (config_index * 100) + fold_index)
        outputs = predict_outputs(model, valid_tensors, str(config["loss_name"]))
        pred = outputs["goal_probability"]
        metrics = metric_row(valid_df[TARGET].to_numpy(), pred, threshold=0.5)
        metrics.update({"model": config["name"], "fold": fold_id})
        fold_rows.append(metrics)
        oof_parts.append(
            valid_df[["scored_after_eval_key", TARGET, "fold"]].assign(
                model=config["name"],
                goal_probability=pred,
                log_lambda_rate=outputs["log_lambda_rate"],
                lambda_rate=outputs["lambda_rate"],
                output_interpretation=np.where(str(config["loss_name"]) == "poisson", "intensity_to_probability", "direct_probability"),
            )
        )

fold_metrics_df = pd.DataFrame(fold_rows).sort_values(["model", "fold"]).reset_index(drop=True)
oof_df = pd.concat(oof_parts, ignore_index=True)

summary_rows = []
threshold_rows = []
for config in FT_CONFIGS:
    model_name = config["name"]
    model_fold_df = fold_metrics_df[fold_metrics_df["model"] == model_name].copy()
    model_oof_df = oof_df[oof_df["model"] == model_name].copy()
    best_threshold = tune_threshold(model_oof_df[TARGET].to_numpy(), model_oof_df["goal_probability"].to_numpy())
    threshold_rows.append({"model": model_name, **best_threshold})
    tuned_metrics = metric_row(model_oof_df[TARGET].to_numpy(), model_oof_df["goal_probability"].to_numpy(), threshold=best_threshold["threshold"])
    summary_rows.append(
        {
            "model": model_name,
            "mean_pr_auc": float(model_fold_df["pr_auc"].mean()),
            "std_pr_auc": float(model_fold_df["pr_auc"].std(ddof=0)),
            "mean_balanced_accuracy_at_0_5": float(model_fold_df["balanced_accuracy"].mean()),
            "mean_roc_auc": float(model_fold_df["roc_auc"].mean()),
            "mean_brier_score": float(model_fold_df["brier_score"].mean()),
            "tuned_threshold": float(best_threshold["threshold"]),
            "oof_balanced_accuracy_at_tuned_threshold": float(tuned_metrics["balanced_accuracy"]),
            "loss_name": config["loss_name"],
            "d_token": int(config["d_token"]),
            "n_layers": int(config["n_layers"]),
            "n_heads": int(config["n_heads"]),
            "dropout": float(config["dropout"]),
            "public_output": "goal_probability",
            "internal_interpretation": "goal_intensity_lambda" if config["loss_name"] == "poisson" else "direct_goal_probability",
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["mean_pr_auc", "oof_balanced_accuracy_at_tuned_threshold"], ascending=[False, False]).reset_index(drop=True)
threshold_df = pd.DataFrame(threshold_rows).sort_values(["balanced_accuracy", "threshold"], ascending=[False, True]).reset_index(drop=True)

fold_metrics_df.to_csv(ARTIFACT_DIR / "fold_metrics.csv", index=False)
oof_df.to_csv(ARTIFACT_DIR / "oof_predictions.csv", index=False)
summary_df.to_csv(ARTIFACT_DIR / "model_summary.csv", index=False)
threshold_df.to_csv(ARTIFACT_DIR / "threshold_summary.csv", index=False)
pd.DataFrame(FT_CONFIGS).to_csv(ARTIFACT_DIR / "config_registry.csv", index=False)

summary_df


Running ft_repro_bce
Running ft_wide_bce
Running ft_wide_focal
Running ft_deep_focal
Running ft_wide_poisson
Running ft_deep_poisson


,model,mean_pr_auc,std_pr_auc,mean_balanced_accuracy_at_0_5,mean_roc_auc,mean_brier_score,tuned_threshold,oof_balanced_accuracy_at_tuned_threshold,loss_name,d_token,n_layers,n_heads,dropout,public_output,internal_interpretation
0,ft_wide_poisson,0.126164,0.041510,0.505864,0.635936,0.055065,0.06,0.593939,poisson,64,3,8,0.10,goal_probability,goal_intensity_lambda
1,ft_deep_focal,0.125044,0.031707,0.612948,0.636911,0.217815,0.48,0.617613,focal,48,4,6,0.15,goal_probability,direct_goal_probability
2,ft_wide_focal,0.117719,0.042528,0.542152,0.631245,0.241740,0.45,0.555262,focal,64,3,8,0.12,goal_probability,direct_goal_probability
3,ft_repro_bce,0.103754,0.021198,0.587557,0.634609,0.240371,0.44,0.598931,bce,32,2,4,0.10,goal_probability,direct_goal_probability
4,ft_wide_bce,0.102832,0.024856,0.598645,0.604998,0.226964,0.41,0.600608,bce,64,3,8,0.10,goal_probability,direct_goal_probability
5,ft_deep_poisson,0.091434,0.034914,0.500000,0.570830,0.104363,0.05,0.527210,poisson,48,4,6,0.15,goal_probability,goal_intensity_lambda
